In [11]:
import random
import numpy as np

class WumpusWorld:
    def __init__(self, grid_size, pit_size):
        self.GRID_SIZE = grid_size
        self.PIT_SIZE = pit_size
        self.grid = [["." for _ in range(self.GRID_SIZE)] for _ in range(self.GRID_SIZE)]
        self.agent_position = (0, 0)
        self.agent_has_gold = False
        self.wumpus_alive = True
        self.place_item("W")
        self.place_item("G")
        for _ in range(self.PIT_SIZE):
            self.place_item("P")

    def place_item(self, item):
        while True:
            x, y = random.randint(0, self.GRID_SIZE - 1), random.randint(0, self.GRID_SIZE - 1)
            if (x == 0 and y == 0):  # Skip agent's starting position
                continue
            if self.grid[x][y] == ".":  # Place only on empty cells
                self.grid[x][y] = item
                break

    def percept(self, position):
        x, y = position
        percepts = []
        for dx, dy in [(1, 0), (0, 1), (-1, 0), (0, -1)]:  # Check adjacent cells
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.GRID_SIZE and 0 <= ny < self.GRID_SIZE:
                if self.grid[nx][ny] == "W":
                    percepts.append("Stench")
                elif self.grid[nx][ny] == "P":
                    percepts.append("Breeze")
        if self.grid[x][y] == "G":
            percepts.append("Glitter")
        return percepts

    def display_grid(self):
        temp_grid = [row[:] for row in self.grid]
        ax, ay = self.agent_position
        temp_grid[ax][ay] = "A"
        print(np.matrix(temp_grid))
        
    def move_agent(self, direction):
        x, y = self.agent_position
        if direction == "up":
            x -= 1
        elif direction == "down":
            x += 1
        elif direction == "left":
            y -= 1
        elif direction == "right":
            y += 1
        if 0 <= x < self.GRID_SIZE and 0 <= y < self.GRID_SIZE:
            self.agent_position = (x, y)
            percepts = self.percept(self.agent_position)
            print(f"Agent moved {direction}. New position: {self.agent_position}. Percepts: {percepts}")
            if "Glitter" in percepts:
                self.agent_has_gold = True
                print("Agent found the gold!")
        else:
            print("Move is out of bounds!")

class Agent:
    def __init__(self):
        self.has_gold = False

    def decide_move(self, grid, percepts, position, grid_size):
        if "Glitter" in percepts:
            print("Agent: Gold found! Grabbing it.")
            self.has_gold = True
            return "grab"
        
        # Determine potential safe moves
        x, y = position
        safe_moves = []
        for dx, dy, direction in [(-1, 0, "up"), (1, 0, "down"), (0, -1, "left"), (0, 1, "right")]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size:
                if grid[nx][ny] != "W" and grid[nx][ny] != "P":  # Avoid hazards
                    safe_moves.append(direction)
        
        # Choose a safe move, or move randomly if none are safe
        if safe_moves:
            return random.choice(safe_moves)
        else:
            return random.choice(["up", "down", "left", "right"])

# Game initialization
GRID_SIZE = 4
PIT_SIZE = 3
world = WumpusWorld(GRID_SIZE, PIT_SIZE)
agent = Agent()

# Game loop
while not agent.has_gold:
    percepts = world.percept(world.agent_position)
    action = agent.decide_move(world.grid, percepts, world.agent_position, GRID_SIZE)
    world.display_grid()
    if action == "grab":
        break
    else:
        world.move_agent(action)

print("Agent has the gold!")


[['A' '.' 'P' '.']
 ['.' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved right. New position: (0, 1). Percepts: ['Breeze', 'Breeze']
[['.' 'A' 'P' '.']
 ['.' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved left. New position: (0, 0). Percepts: []
[['A' '.' 'P' '.']
 ['.' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved right. New position: (0, 1). Percepts: ['Breeze', 'Breeze']
[['.' 'A' 'P' '.']
 ['.' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved left. New position: (0, 0). Percepts: []
[['A' '.' 'P' '.']
 ['.' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved down. New position: (1, 0). Percepts: ['Breeze']
[['.' '.' 'P' '.']
 ['A' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved up. New position: (0, 0). Percepts: []
[['A' '.' 'P' '.']
 ['.' 'P' '.' '.']
 ['.' 'P' 'W' '.']
 ['G' '.' '.' '.']]
Agent moved down. New position: (1, 0). Percepts: ['Breeze']
[['.' '.' 'P' '.']
 ['A' 'P' '.' '.']
